In [2]:
from Crypto . Cipher import AES
from Crypto . Random import get_random_bytes
import json
def pad ( data , block_size =16) :
  pad_len = block_size - ( len ( data ) % block_size )
  return data + bytes ([ pad_len ]) * pad_len
def read_bmp ( file_path ):
  with open ( file_path , "rb") as f:bmp = f. read ()
  header = bmp [:54]
  body = bmp [54:]
  return header , body
def write_bmp ( file_path , header , body ):
  with open ( file_path , "wb") as f:f. write ( header + body )





In [3]:
def generate_crypto_material () :
  material = {
  "key": get_random_bytes (16) . hex () , # AES -128
  "iv": get_random_bytes (16) .hex () , # para CBC
  "nonce": get_random_bytes (8) . hex () # para CTR
  }
  return material
def save_crypto_material ( material , output_file ):
  with open ( output_file , "w") as f :json . dump ( material , f , indent =4)
def load_crypto_material ( input_file ):
  with open ( input_file , "r") as f:material = json . load ( f)
  return material


In [4]:
material = generate_crypto_material ()
save_crypto_material ( material , "mis_claves_31624.json")
print ( material )


{'key': '97112cbfd11d394a202980a80727c93a', 'iv': 'bfcd400dcabec57126e301bda4a001c1', 'nonce': '1b123ef3009c8632'}


In [5]:
def encrypt_bmp(input_file, output_file, mode_name, material):
    header, body = read_bmp(input_file)
    key = bytes.fromhex(material["key"])

    if mode_name == "ECB":
        cipher = AES.new(key, AES.MODE_ECB)
        body_padded = pad(body)
        encrypted = cipher.encrypt(body_padded)
        encrypted = encrypted[:len(body)]
    elif mode_name == "CBC":
        iv = bytes.fromhex(material["iv"])
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        body_padded = pad(body)
        encrypted = cipher.encrypt(body_padded)
        encrypted = encrypted[:len(body)]
    elif mode_name == "CTR":
        nonce = bytes.fromhex(material["nonce"])
        cipher = AES.new(key, AES.MODE_CTR, nonce=nonce)
        encrypted = cipher.encrypt(body)
    else:
        raise ValueError("Modo no soportado")

    write_bmp(output_file, header, encrypted)
    print(f"{mode_name} cifrado guardado en: {output_file}")

In [6]:
material = load_crypto_material ("mis_claves_31624.json")
encrypt_bmp ("android.bmp", "ecb.bmp", "ECB", material )
encrypt_bmp ("android.bmp", "cbc.bmp", "CBC", material )
encrypt_bmp ("android.bmp", "ctr.bmp", "CTR", material )


ECB cifrado guardado en: ecb.bmp
CBC cifrado guardado en: cbc.bmp
CTR cifrado guardado en: ctr.bmp


In [7]:
def decrypt_bmp ( input_file , output_file , mode_name , material ):
  header , body = read_bmp ( input_file )
  key = bytes . fromhex ( material ["key"])

  if mode_name == "ECB":
    cipher = AES . new ( key , AES . MODE_ECB )
    decrypted = cipher . decrypt ( pad ( body ))
    decrypted = decrypted [: len ( body ) ]

  elif mode_name == "CBC":
    iv = bytes . fromhex ( material ["iv"])
    cipher = AES . new ( key , AES . MODE_CBC , iv = iv )
    decrypted = cipher . decrypt ( pad ( body ))
    decrypted = decrypted [: len ( body ) ]

  elif mode_name == "CTR":
    nonce = bytes . fromhex ( material ["nonce"])
    cipher = AES . new ( key , AES . MODE_CTR , nonce = nonce )
    decrypted = cipher . decrypt ( body )

  else :
    raise ValueError ("Modo no soportado")


  write_bmp ( output_file , header , decrypted )
  print (f"{ mode_name } descifrado guardado en: { output_file }")


In [8]:
material = load_crypto_material ("mis_claves_31624.json")
decrypt_bmp ("ecb.bmp", "ecb_descifrada.bmp", "ECB", material )
decrypt_bmp ("cbc.bmp", "cbc_descifrada.bmp", "CBC", material )
decrypt_bmp ("ctr.bmp", "ctr_descifrada.bmp", "CTR", material )

ECB descifrado guardado en: ecb_descifrada.bmp
CBC descifrado guardado en: cbc_descifrada.bmp
CTR descifrado guardado en: ctr_descifrada.bmp


In [9]:
def flip_byte ( file_path , output_path , position ):
  with open ( file_path , "rb") as f:
    data = bytearray (f . read () )
  print ("Byte original :", data [ position ])
  data [ position ] ^= 0xFF
  print ("Byte modificado :", data [ position ])
  with open ( output_path , "wb") as f:
    f. write ( data )
    print ("Archivo modificado guardado en:", output_path )

In [10]:
flip_byte ("ecb.bmp", "ecb_mod.bmp", 2000)
flip_byte ("cbc.bmp", "cbc_mod.bmp", 2000)
flip_byte ("ctr.bmp", "ctr_mod.bmp", 2000)


Byte original : 110
Byte modificado : 145
Archivo modificado guardado en: ecb_mod.bmp
Byte original : 185
Byte modificado : 70
Archivo modificado guardado en: cbc_mod.bmp
Byte original : 118
Byte modificado : 137
Archivo modificado guardado en: ctr_mod.bmp


In [11]:
material = load_crypto_material ("mis_claves_31624.json")
decrypt_bmp ("ecb_mod.bmp", "ecb_mod_descifrada.bmp", "ECB", material )
decrypt_bmp ("cbc_mod.bmp", "cbc_mod_descifrada.bmp", "CBC", material )
decrypt_bmp ("ctr_mod.bmp", "ctr_mod_descifrada.bmp", "CTR", material )

ECB descifrado guardado en: ecb_mod_descifrada.bmp
CBC descifrado guardado en: cbc_mod_descifrada.bmp
CTR descifrado guardado en: ctr_mod_descifrada.bmp


### **COMPANERO**

In [110]:
material_companero = load_crypto_material ("claves_companero.json")
decrypt_bmp ("imagen_companero_cifrada.bmp","imagen_companero_descifrada.bmp","CBC", material_companero)

CBC descifrado guardado en: imagen_companero_descifrada.bmp


In [111]:
mis_materiales = load_crypto_material ("mis_claves_34887.json")
decrypt_bmp ("imagen_companero_cifrada.bmp","fallo_descifrado.bmp","CBC", mis_materiales )


CBC descifrado guardado en: fallo_descifrado.bmp
